# 01 — Hyrje, të dhënat dhe validimi

## Konteksti dhe problemi shkencor

Do të ndërtojmë një eksperiment të vogël klasifikimi me pulse detektori për të kuptuar strukturën e një workflow-je ML pa e ngatërruar metodologjinë me algoritmin.

### Objektivat e të nxënit

- të dalloni veçoritë, etiketat, parametrat dhe hiperparametrat;
- të përdorni train/validation/test dhe cross-validation;
- të dalloni rrjedhjen e informacionit;
- të përdorni pipeline për preprocessing.

> **Parimi i laboratorit:** matematikë → implementim nga parimet e para → implementim me bibliotekë → krahasim numerik → interpretim fizik.


In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, KFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, Perceptron, SGDRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
DATA = Path("../datasets/synthetic")
SRC = Path("../src").resolve()
if str(SRC.parent) not in sys.path:
    sys.path.insert(0, str(SRC.parent))
np.set_printoptions(precision=4, suppress=True)
RANDOM_STATE = 42


## 1. Nga pyetja fizike te tabela e të dhënave

Një problem ML nuk fillon me `sklearn`, por me një pyetje. Në këtë notebook përdorim pulse sintetike detektori. Çdo rresht është një ngjarje; amplituda dhe gjerësia janë **veçori (features)**, ndërsa `class_signal` është **etiketa (label)**.

In [ ]:
df=pd.read_csv(DATA/'detector_pulses.csv'); df.head()

In [ ]:
df.describe(include='all')

## 2. Vizualizimi fillestar dhe kontrolli fizik

Shikojmë shpërndarjet para se të modelojmë. Një ndarje e dukshme në dy dimensione na jep intuitë, por nuk duhet të përdoret si zëvendësim për validimin.

In [ ]:
plt.figure(figsize=(6,4.5)); plt.scatter(df.amplitude,df.width_us,c=df.class_signal,s=15,alpha=.6); plt.xlabel('Amplituda'); plt.ylabel('Gjerësia [μs]'); plt.show()

## 3. Trajnim dhe provë

Përdorim një model të thjeshtë vetëm për të studiuar procedurën. Në këtë temë modeli është dytësor; fokusi është ndarja, standardizimi dhe rrjedhja e informacionit.

In [ ]:
from sklearn.linear_model import LogisticRegression
X=df[['amplitude','width_us']]; y=df.class_signal
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=RANDOM_STATE)
pipe=make_pipeline(StandardScaler(),LogisticRegression())
pipe.fit(X_train,y_train)
print('train=',pipe.score(X_train,y_train),'test=',pipe.score(X_test,y_test))

## 4. Validimi i kryqëzuar

Një split i vetëm është vetëm një realizim. Në $K$-fold cross-validation raportojmë mesataren **dhe** variabilitetin.

In [ ]:
cv=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE)
r=cross_validate(pipe,X,y,cv=cv,return_train_score=True)
pd.DataFrame(r)[['train_score','test_score']]

In [ ]:
print(f"CV: {r['test_score'].mean():.3f} ± {r['test_score'].std():.3f}")

## 5. Eksperiment: rrjedhje informacioni

Krijojmë një shembull ku standardizimi kryhet gabimisht para cross-validation. Në këtë dataset efekti mund të jetë i vogël; pikërisht kjo e bën gabimin të rrezikshëm: kodi duket i rregullt dhe rezultati mund të ndryshojë vetëm pak. Parimi mbetet i gabuar.

In [ ]:
from sklearn.model_selection import cross_val_score
sc=StandardScaler(); X_leaky=sc.fit_transform(X)
score_bad=cross_val_score(LogisticRegression(),X_leaky,y,cv=cv)
score_good=cross_val_score(pipe,X,y,cv=cv)
print('gabim:',score_bad.mean(), 'pipeline:',score_good.mean())

## 6. Eksperiment më i fortë: përzgjedhje veçorish para ndarjes

Shtojmë shumë veçori zhurme dhe zgjedhim veçoritë më të korreluara me etiketën duke përdorur **të gjithë** dataset-in. Kjo krijon rrjedhje më të dukshme.

In [ ]:
rng=np.random.default_rng(3); Z=rng.normal(size=(len(y),200)); tmp=pd.DataFrame(Z); corr=np.abs(np.corrcoef(np.c_[Z,y],rowvar=False)[-1,:-1]); idx=np.argsort(corr)[-8:]; X_bad=Z[:,idx]
print('CV pas përzgjedhjes së gabuar:',cross_val_score(LogisticRegression(max_iter=2000),X_bad,y,cv=cv).mean())

## 7. Interpretim

Në një eksperiment real, ``rreshti'' mund të mos jetë njësi e pavarur. Pulse nga i njëjti run, spektra nga i njëjti kampion ose dritare të mbivendosura nga e njëjta seri kohore duhet të ndahen sipas grupeve, jo rastësisht.

## 8. Ushtrim studenti

1. Ndryshoni `test_size` dhe studioni variabilitetin e rezultatit.
2. Përsëritni 30 train/test split-e me fara të ndryshme dhe vizatoni histogramin e accuracy.
3. Shpjegoni me fjalët tuaja ndryshimin midis parametrit dhe hiperparametrit.
4. Propozoni një ndarje korrekte për matje të marra në 10 ditë eksperimentale.

## 9. Shembull i plotë i punuar

Ndërtoni një pipeline, bëni 5-fold CV, pastaj rezervoni 20% si test final dhe raportoni score-n vetëm një herë. Shënoni qartë cilat vendime u morën para se të hapej testi.

In [ ]:
X_dev,X_final,y_dev,y_final=train_test_split(X,y,test_size=.2,stratify=y,random_state=11)
print('CV development:',cross_val_score(pipe,X_dev,y_dev,cv=5).mean())
pipe.fit(X_dev,y_dev); print('Final test:',pipe.score(X_final,y_final))

## 10. Playground

Eksperimentoni me një veçori të tretë zhurme, me `MinMaxScaler`, ose me një ndarje ku 90% e klasës 1 futet në trajnim dhe diskutoni pse rezultati nuk është i krahasueshëm.